# 📊 Análise de E-commerce com SQL (SQLite)

## 💼 Problema de Negócio
Analisar dados de um e-commerce para entender padrões de receita, comportamento dos clientes e eficiência da entrega.

## 🎯 Objetivo
Aplicar consultas SQL em um banco de dados real para extrair insights básicos de negócio.

## 📚 Contexto
Projeto desenvolvido durante meus estudos em Data Science pela Alura.

Utilizei Inteligência Artificial como apoio para organização e revisão das análises, garantindo entendimento durante o processo.


## 🧠 Estrutura dos Dados

O banco de dados é composto por múltiplas tabelas relacionadas:

- customers: informações dos clientes
- orders: dados dos pedidos
- order_payments: informações de pagamento

Essas tabelas são conectadas através de identificadores, permitindo análises mais completas.

In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/content/olist.sqlite')

pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,product_category_name_translation
1,sellers
2,customers
3,geolocation
4,order_items
5,order_payments
6,order_reviews
7,orders
8,products
9,leads_qualified


## 🔍 Exploração dos dados

Visualizando as principais tabelas para entender a estrutura do banco.

In [3]:
pd.read_sql_query("SELECT * FROM orders LIMIT 5;", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
pd.read_sql_query("SELECT * FROM customers LIMIT 5;", conn)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
pd.read_sql_query("SELECT * FROM order_payments LIMIT 5;", conn)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## 💰 Receita Total

A receita total indica o volume financeiro movimentado pela plataforma, sendo um indicador chave de desempenho do negócio.

In [6]:
query = """
SELECT ROUND(SUM(payment_value), 2) AS total_revenue
FROM order_payments
"""
df = pd.read_sql_query(query, conn)
df

,total_revenue
0,16008872.12


A receita total representa o volume financeiro movimentado pela plataforma.

## 🌎 Receita por Estado

São Paulo concentra a maior parte da receita, indicando maior presença de mercado e potencial concentração de clientes na região.

In [7]:
query = """
SELECT
    c.customer_state,
    ROUND(SUM(p.payment_value), 2) AS revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_payments p ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY revenue DESC
"""
df = pd.read_sql_query(query, conn)
df

,customer_state,revenue
0,SP,5998226.96
1,RJ,2144379.69
2,MG,1872257.26
3,RS,890898.54
4,PR,811156.38
5,SC,623086.43
6,BA,616645.82
7,DF,355141.08
8,GO,350092.31
9,ES,325967.55


Observa-se concentração de receita em determinados estados, com destaque para São Paulo.

## 🚚 Tempo Médio de Entrega

O tempo médio de entrega é um indicador importante da eficiência logística, podendo impactar diretamente a satisfação e retenção de clientes.

In [8]:
query = """
SELECT
    ROUND(AVG(
        JULIANDAY(order_delivered_customer_date) -
        JULIANDAY(order_purchase_timestamp)
    ), 2) AS avg_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
df

,avg_delivery_days
0,12.56


O tempo médio de entrega é um indicador importante da eficiência logística.

Prazos mais longos podem impactar a satisfação do cliente e influenciar futuras compras.

## 👤 Clientes com mais pedidos

A presença de clientes com múltiplos pedidos indica comportamento recorrente, o que pode representar maior valor ao longo do tempo para o negócio.

In [9]:
query = """
SELECT
    customer_id,
    COUNT(order_id) AS total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 10
"""
df = pd.read_sql_query(query, conn)
df

,customer_id,total_orders
0,ffffe8b65bbe3087b653a978c870db99,1
1,ffffa3172527f765de70084a7e53aae8,1
2,ffff42319e9b2d713724ae527742af25,1
3,fffeda5b6d849fbd39689bb92087f431,1
4,fffecc9f79fd8c764f843e9951b11341,1
5,fffcb937e9dd47a13f05ecb8290f4d3e,1
6,fffc22669ca576ae3f654ea64c8f36be,1
7,fffb97495f78be80e2759335275df2aa,1
8,fffa0238b217e18a8adeeda0669923a3,1
9,fff93c1da78dafaaa304ff032abc6205,1


Os dados mostram que alguns clientes realizam múltiplos pedidos, indicando comportamento recorrente.

Esse tipo de cliente pode representar maior valor ao longo do tempo e ser relevante para estratégias de fidelização.

## 📌 Conclusão

A análise permitiu identificar padrões básicos de funcionamento do e-commerce, incluindo distribuição de receita, comportamento de clientes e eficiência logística.

Os resultados mostram concentração de receita em determinadas regiões, presença de clientes recorrentes e a importância do tempo de entrega na experiência do cliente.

## 📈 Aprendizados

- Uso de JOIN para integrar múltiplas tabelas
- Aplicação de funções agregadas em SQL
- Interpretação de dados com foco em negócio
- Estruturação de análise em ambiente de notebook